# 🥒 Zucchini CNN Regression — DNG RAW Encoding Ablation (sRGB vs Linear vs ACES)

**Pipeline:** `DNG raw → RAW-to-RGB encoding → Statistical LAB filter (K=2.0σ) → LAB stack → RegressionCNN → score`

This notebook keeps the **best pre-processing identified in `zucchini_cnn_colorspace_filtering_ablation.ipynb`**
— **LAB color space** + **Statistical filter (K=2.0σ)** — and now trains the **exact `RegressionCNN`
architecture and training recipe from `260305_cnn_regression_analysis.ipynb`** instead of the earlier
GAP-based `CNNRegressor`:

- **Model**: 6 conv blocks (16→32→64→128→256→512 channels, the last block with a single conv instead of
  two), flatten (not global-average-pool) into a 3-layer FC head, **ELU** output activation.
- **Loss / optimizer**: `MSELoss`, Adam (`lr=1e-3`, no weight decay), `ReduceLROnPlateau`
  (factor 0.1, patience 5 on validation loss), early stopping on validation loss (patience 12).
- **Augmentation**: random H/V flips + random 90°-multiple rotations + `Normalize(mean=0.5, std=0.5)`,
  plus Gaussian **target-noise regularization** (`std=0.15`) added to training targets only — all lifted
  directly from the reference notebook's `ImageRegressionDataset`/transform pipeline.

Two things changed from the reference notebook to fit this pipeline:
1. It loaded pre-existing 16-bit `.tif` files directly; here the RGB comes from **demosaicing raw `.dng`
   files** via `rawpy`, through 3 candidate RAW→RGB encodings (this notebook's ablation axis).
2. It used one 80/20 train/val split; here we keep the **5-fold stratified CV** from the color-space
   ablation notebook so results are directly comparable across notebooks and not a single split's luck.

**New ablation axis — RAW→RGB encoding method**: since the DNG contains only raw sensor data, *something*
has to decide how that raw data becomes an RGB image before the CNN ever sees it. This notebook trains and
compares three encodings:
- **sRGB** — the conventional display-referred encoding (sRGB primaries + sRGB tone curve).
- **Linear** — scene-referred, no tone curve (gamma = 1), only a percentile-based exposure normalization.
- **ACES** — camera-native RGB → ACES2065-1 (AP0 primaries) → a filmic tone-map standing in for the ACES
  RRT+ODT (see the caveat in Section 3 — this is an **approximation**, not a certified ACES/OCIO pipeline).

**Color space and filter are held fixed** at LAB + Statistical (K=2.0σ) for all three encodings, so any
metric difference is attributable to the encoding step alone.

⚠️ **Compute note**: 3 encodings × 5 folds = 15 training runs, plus one full raw-demosaic pass per encoding
(cached to disk, so re-running the notebook won't redo finished work unless `FORCE_RETRAIN = True`).


## 0 · Imports & Setup

In [ ]:
# This notebook needs `rawpy` (a Python wrapper around LibRaw) to decode .dng files.
# If it isn't installed yet:
#     pip install rawpy
import os, glob, random, warnings, time, pickle
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import spearmanr, kendalltau

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

import rawpy

# ── Reproducibility ──────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ── Device ────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Torch device : {device}')
if device.type == 'cuda':
    torch.cuda.manual_seed(SEED)
    print(f'GPU          : {torch.cuda.get_device_name(0)}')
print(f'torch        : {torch.__version__}')
print(f'rawpy        : {rawpy.__version__}')


## 1 · Configuration

In [ ]:
# ── EDIT THIS PATH — root folder that contains the dated scoring session folders ──
DNG_BASE_PATH = r"F:\\Aymen\\server\\Database_CSVS\\CSVS_Zucchini_202602_LaMojonena\\Zucchini_COLOR_Evaluation\\Zucchini_GREEN_FOR_SPAIN_on_LVS_platform"
DNG_FOLDER_PATTERN = "2026-02-26_Scorings_"
# ───────────────────────────────────────────────────────────────────────────

IMG_SIZE      = (64, 64)     # matches the tested config of RegressionCNN (depth=16 <-> 64x64)
BATCH_SIZE    = 32           # from the reference notebook
K_FOLDS       = 5
SEED          = 42
FG_THRESH     = 10           # foreground threshold — same convention as the TIFF notebooks
K_STAT        = 2.0          # sigma half-width for the statistical LAB filter — BEST method per ablation

# Fixed "best config" carried over from zucchini_cnn_colorspace_filtering_ablation.ipynb.
# The only thing being ablated in THIS notebook is the RAW->RGB encoding method below.
FIXED_COLOR_SPACE = 'LAB'
FIXED_FILTER_NAME = f'Statistical (K={K_STAT:.1f}\u03c3)'

ENCODING_METHODS = ['sRGB', 'Linear', 'ACES']

# ── RegressionCNN + training recipe, copied from 260305_cnn_regression_analysis.ipynb ──
CNN_DEPTH        = 16        # base channel depth (16->32->64->128->256->512 through the 6 blocks)
EPOCHS           = 1000      # ceiling — early stopping (patience below) will almost always stop first
LR_INIT          = 1e-3
LR_FACTOR        = 0.1       # ReduceLROnPlateau factor
PATIENCE_LR      = 5         # ReduceLROnPlateau patience (on validation loss)
PATIENCE_ES      = 12        # early-stopping patience (on validation loss)
TARGET_NOISE_STD = 0.15      # Gaussian noise added to training targets only (regularization)

SCORE_FOLDERS = list(range(1, 10))       # folders 1-9 -> scores 1-9
NUM_CLASSES   = 9
IDX_TO_SCORE  = np.arange(1, 10, dtype=np.float32)   # 1..9

CACHE_DIR = 'dng_cnn_cache'
OUT_DIR   = 'dng_cnn_outputs'
CKPT_DIR  = 'dng_cnn_checkpoints'
for d in (CACHE_DIR, OUT_DIR, CKPT_DIR):
    os.makedirs(d, exist_ok=True)

FORCE_RETRAIN = False   # set True to ignore all caches (raw decode + OOF results) and redo everything

def savefig(name, dpi=150):
    path = os.path.join(OUT_DIR, name)
    plt.savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'  \U0001f4be {path}')

print(f'DNG_BASE_PATH      : {DNG_BASE_PATH}')
print(f'DNG_FOLDER_PATTERN : {DNG_FOLDER_PATTERN}*')
print(f'Fixed config       : {FIXED_COLOR_SPACE} + {FIXED_FILTER_NAME}')
print(f'Encoding methods   : {ENCODING_METHODS}')
print(f'Model              : RegressionCNN (depth={CNN_DEPTH}, img={IMG_SIZE})')


## 2 · Locate DNG Session Folders & Collect File Paths

The raw pixel data is decoded once per encoding (Section 3), but the file paths and score labels are
encoding-independent, so they're collected once here. This mirrors the `Score=NN` subfolder convention
used by the TIFF `find_and_filter_images()` loader — adjust the `candidates` list below if the DNG capture
folders use a different naming scheme than the nocc output.

In [ ]:
def find_session_folders(base_path, pattern):
    '''Finds every folder directly under base_path whose name starts with `pattern`
    (e.g. every dated "2026-02-26_Scorings_*" capture session). Falls back to
    treating base_path itself as the session folder if no match is found.'''
    if not os.path.exists(base_path):
        raise FileNotFoundError(f'DNG_BASE_PATH not found: {base_path}')
    candidates = sorted(glob.glob(os.path.join(base_path, f'{pattern}*')))
    session_folders = [c for c in candidates if os.path.isdir(c)]
    if not session_folders:
        print(f'  \u26a0 No folders matched "{pattern}*" directly under base_path — '
              f'falling back to treating DNG_BASE_PATH itself as the session folder.')
        session_folders = [base_path]
    return session_folders


def collect_dng_paths(base_path, pattern, score_folders):
    '''Walk every matched session folder for Score=NN (or plain NN) subfolders and
    gather .dng file paths per score — same folder convention as the TIFF pipeline's
    find_and_filter_images(), just pointed at raw .dng files instead of processed .tif files.'''
    session_folders = find_session_folders(base_path, pattern)
    print(f'Session folders matched ({len(session_folders)}):')
    for s in session_folders:
        print(f'    {s}')

    path_map = {s: [] for s in score_folders}
    for session in session_folders:
        for folder_num in score_folders:
            candidates = [
                os.path.join(session, str(folder_num)),
                os.path.join(session, f'Score={folder_num:02d}'),
                os.path.join(session, f'Score={folder_num}'),
                os.path.join(session, f'{folder_num:02d}'),
            ]
            folder_path = next((p for p in candidates if os.path.isdir(p)), None)
            if folder_path is None:
                continue
            dngs = (glob.glob(os.path.join(folder_path, '*.dng'))
                  + glob.glob(os.path.join(folder_path, '*.DNG')))
            path_map[folder_num].extend(sorted(set(dngs)))

    print()
    for k in score_folders:
        print(f'  [Score {k}] {len(path_map[k])} DNG files found')
    return path_map


score_to_paths = collect_dng_paths(DNG_BASE_PATH, DNG_FOLDER_PATTERN, SCORE_FOLDERS)

flat_paths, flat_labels = [], []
for folder_num in SCORE_FOLDERS:
    for p in score_to_paths[folder_num]:
        flat_paths.append(p)
        flat_labels.append(folder_num - 1)   # 0-indexed

if len(flat_paths) == 0:
    raise RuntimeError('No DNG files found! Check DNG_BASE_PATH / DNG_FOLDER_PATTERN and the '
                        'Score=NN subfolder naming assumption above.')

y_raw = np.array(flat_labels, dtype=np.int32)
y = IDX_TO_SCORE[y_raw]
print(f'\n\u2714 Total DNG files: {len(flat_paths)} across {len(set(flat_labels))} score classes')


## 3 · RAW → RGB Encoding Functions (sRGB / Linear / ACES)

Each `.dng` is demosaiced once per encoding via `rawpy`/LibRaw. The three encodings differ only in what
happens **after** demosaicing:

- **sRGB**: LibRaw's own sRGB output color space + the exact sRGB tone curve (piecewise linear + power,
  `gamma=(2.4, 12.92)`).
- **Linear**: same sRGB primaries, but `gamma=(1, 1)` — i.e. no tone curve at all — then a percentile-based
  exposure stretch (there's no tone curve to compress highlights, so without this the image would be almost
  entirely clipped white or near-black).
- **ACES**: converts camera-native linear RGB into ACES2065-1 (the AP0 primaries used by the Academy Color
  Encoding System), then applies a filmic tone curve as a stand-in for the ACES RRT+ODT.

**Honesty note on the ACES path**: a certified ACES pipeline runs its Reference Rendering Transform and
Output Device Transform through OpenColorIO/CTL, using the official ACES config. Here, when the installed
LibRaw build exposes a native `ACES` color space we use that for the camera→ACES2065-1 step; otherwise we
fall back to a manual camera→XYZ→ACES2065-1 matrix transform. Either way, the *tone-mapping* step (turning
scene-linear ACES values into something displayable/trainable) uses the widely-used Narkowicz (2015) fast
filmic-curve fit, not the full reference ODT. Treat the "ACES" result here as **ACES-style**, not
bit-exact — if the comparison looks promising, it's worth validating against a real OCIO/`ctlrender` pass
before drawing final conclusions.

In [ ]:
HAS_NATIVE_ACES = hasattr(rawpy.ColorSpace, 'ACES')
print(f'LibRaw exposes a native ACES color space: {HAS_NATIVE_ACES}')

# Fixed XYZ (D60) -> ACES2065-1 (AP0 primaries) matrix, per the ACES specification —
# used only in the manual fallback path when rawpy.ColorSpace.ACES isn't available.
XYZ_TO_ACES2065_1 = np.array([
    [ 1.0498110175, 0.0000000000, -0.0000974845],
    [-0.4959030231, 1.3733130458,  0.0982400361],
    [ 0.0000000000, 0.0000000000,  0.9912520182],
], dtype=np.float64)


def _percentile_normalize(x, pct=99.5, eps=1e-4):
    '''Scene-linear values have no built-in exposure reference, so we approximate one
    from image statistics (a high percentile) rather than a light-meter/gray-card
    reading. This is a pragmatic stand-in, not a radiometric calibration.'''
    hi = max(np.percentile(x, pct), eps)
    return np.clip(x / hi, 0, None)


def _aces_rrt_odt_approx(x):
    '''Narkowicz (2015) fast ACES filmic curve — a well-known approximation of the
    combined ACES RRT + sRGB ODT look (highlight roll-off, filmic contrast), not the
    exact CTL/OCIO reference transform.'''
    a, b, c, d, e = 2.51, 0.03, 2.43, 0.59, 0.14
    return np.clip((x * (a * x + b)) / (x * (c * x + d) + e), 0, 1)


def _linear_to_display_uint8(lin16, pct=99.5):
    lin = lin16.astype(np.float32) / 65535.0
    out = _percentile_normalize(lin, pct=pct)
    return (np.clip(out, 0, 1) * 255).astype(np.uint8)


def _aces_encode(raw):
    if HAS_NATIVE_ACES:
        aces = raw.postprocess(
            output_color=rawpy.ColorSpace.ACES,
            gamma=(1, 1), no_auto_bright=True, use_camera_wb=True, output_bps=16,
        ).astype(np.float64) / 65535.0
    else:
        # Manual fallback: camera-native linear RGB -> XYZ -> ACES2065-1
        cam_lin = raw.postprocess(
            output_color=rawpy.ColorSpace.raw,   # keep camera-native primaries, no matrixing
            gamma=(1, 1), no_auto_bright=True, use_camera_wb=True, output_bps=16,
        ).astype(np.float64) / 65535.0

        # LibRaw's rgb_xyz_matrix stores the XYZ->camera matrix (cam_xyz); invert it
        # to get camera->XYZ. Cameras with >3 raw channels report a 4x3 matrix — use
        # the first 3 rows/cols only.
        xyz_to_cam = np.array(raw.rgb_xyz_matrix[:3, :3], dtype=np.float64)
        cam_to_xyz = np.linalg.pinv(xyz_to_cam)

        h, w, _ = cam_lin.shape
        xyz = cam_lin.reshape(-1, 3) @ cam_to_xyz.T
        aces = (xyz @ XYZ_TO_ACES2065_1.T).reshape(h, w, 3)

    aces = _percentile_normalize(aces, pct=99.5)
    disp = _aces_rrt_odt_approx(aces)
    return (disp * 255).astype(np.uint8)


def dng_to_rgb(path, size, encoding):
    with rawpy.imread(path) as raw:
        if encoding == 'sRGB':
            rgb8 = raw.postprocess(
                output_color=rawpy.ColorSpace.sRGB,
                gamma=(2.4, 12.92),          # exact sRGB opto-electronic transfer function
                no_auto_bright=True,
                use_camera_wb=True,
                output_bps=8,
            )
        elif encoding == 'Linear':
            lin16 = raw.postprocess(
                output_color=rawpy.ColorSpace.sRGB,   # sRGB primaries, scene-linear light
                gamma=(1, 1),
                no_auto_bright=True,
                use_camera_wb=True,
                output_bps=16,
            )
            rgb8 = _linear_to_display_uint8(lin16)
        elif encoding == 'ACES':
            rgb8 = _aces_encode(raw)
        else:
            raise ValueError(f'Unknown encoding: {encoding}')

    return cv2.resize(rgb8, (size[1], size[0]), interpolation=cv2.INTER_AREA)


def load_encoded_dataset(paths, size, encoding):
    '''Demosaics every DNG under one encoding and caches the (N,H,W,3) uint8 array to disk —
    this is the expensive step (raw decode), done once per encoding, not once per fold.'''
    cache_path = os.path.join(CACHE_DIR, f'dng_rgb_{encoding}.npy')
    if os.path.exists(cache_path) and not FORCE_RETRAIN:
        arr = np.load(cache_path)
        print(f'[{encoding}] loaded cached RGB array: {arr.shape}')
        return arr

    t0 = time.time()
    imgs, n_failed = [], 0
    for p in paths:
        try:
            imgs.append(dng_to_rgb(p, size, encoding))
        except Exception as e:
            print(f'  \u26a0 Failed {os.path.basename(p)} [{encoding}]: {e}')
            imgs.append(np.zeros((*size, 3), dtype=np.uint8))   # keeps index alignment with y_raw
            n_failed += 1
    arr = np.array(imgs, dtype=np.uint8)
    np.save(cache_path, arr)
    print(f'[{encoding}] demosaiced {len(paths)} DNGs -> {arr.shape}  '
          f'({n_failed} failed)  ({time.time()-t0:.1f}s)')
    if n_failed:
        print(f'  \u26a0 {n_failed} images failed to decode and were zero-filled — '
              f'check the warnings above before trusting this encoding\'s metrics.')
    return arr


## 4 · Statistical LAB Filter (Fixed — K = 2.0σ, the Best Method from the TIFF Ablation)

Identical logic to `stat_filter()` in `zucchini_cnn_colorspace_filtering_ablation.ipynb`. Filtered-out
pixels are set to white (255).

**Assumption worth checking**: the TIFF pipeline's `nocc` images were already background-masked/cropped
before this filter ever saw them. These freshly-demosaiced DNGs have **not** gone through that
crop/mask step — they're closer to full raw captures. `FG_THRESH` alone may not isolate the zucchini as
cleanly here as it did on the pre-cropped TIFFs. Section 8's sample-image grid lets you visually confirm
the filter is behaving sensibly on this data before trusting the downstream metrics.

In [ ]:
def stat_filter(img_rgb, k=K_STAT):
    H, W = img_rgb.shape[:2]
    fg_mask = np.any(img_rgb > FG_THRESH, axis=-1)
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    L_img = lab[..., 0] * 100.0 / 255.0
    a_img = lab[..., 1] - 128.0
    b_img = lab[..., 2] - 128.0
    coords = np.argwhere(fg_mask)
    if len(coords) == 0:
        return {'out': img_rgb.copy(), 'kept_mask': fg_mask, 'kept_pct': 100.0}
    r, c = coords[:, 0], coords[:, 1]
    keep_px = np.ones(len(coords), dtype=bool)
    for ch_img in [L_img, a_img, b_img]:
        vals = ch_img[r, c]
        mu, sd = vals.mean(), vals.std()
        keep_px &= (vals >= mu - k * sd) & (vals <= mu + k * sd)
    kept_mask = np.zeros((H, W), dtype=bool)
    kept_mask[r[keep_px], c[keep_px]] = True
    out = np.full_like(img_rgb, 255)
    out[kept_mask] = img_rgb[kept_mask]
    return {'out': out, 'kept_mask': kept_mask, 'kept_pct': float(keep_px.sum() / len(coords) * 100)}


def apply_stat_filter(images, tag):
    cache_path = os.path.join(CACHE_DIR, f'filtered_{tag}.npy')
    if os.path.exists(cache_path) and not FORCE_RETRAIN:
        arr = np.load(cache_path)
        print(f'[{tag}] loaded cached filtered stack: {arr.shape}')
        return arr
    t0 = time.time()
    res = [stat_filter(img, k=K_STAT) for img in images]
    arr = np.array([r['out'] for r in res], dtype=np.uint8)
    pcts = np.array([r['kept_pct'] for r in res], dtype=np.float32)
    np.save(cache_path, arr)
    print(f'[{tag}] filtered {len(images)} images, kept mean={pcts.mean():.1f}% '
          f'std={pcts.std():.1f}%  ({time.time()-t0:.1f}s)')
    return arr


## 5 · LAB Color-Space Stack (Fixed — the Best Color Space from the TIFF Ablation)

Every filtered RGB image is converted to LAB and channel-stacked into a `(H, W, 3)` uint8 array — the
CNN's only input representation in this notebook.

In [ ]:
def build_lab_stack(images, tag):
    cache_path = os.path.join(CACHE_DIR, f'labstack_{tag}.npy')
    if os.path.exists(cache_path) and not FORCE_RETRAIN:
        arr = np.load(cache_path)
        print(f'[{tag}] loaded cached LAB stack: {arr.shape}')
        return arr
    t0 = time.time()
    arr = np.array([cv2.cvtColor(img, cv2.COLOR_RGB2LAB) for img in images], dtype=np.uint8)
    np.save(cache_path, arr)
    print(f'[{tag}] built LAB stack: {arr.shape}  ({time.time()-t0:.1f}s)')
    return arr


## 6 · 5-Fold Stratified Cross-Validation Splits

Identical `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` scheme as the TIFF notebooks (the
reference `RegressionCNN` notebook used a single stratified 80/20 split — this notebook uses 5-fold CV
instead so results stay comparable with the color-space/filter ablation study). The same fold assignments
are reused for all three encodings, so any metric difference is not a fold-luck artifact.

In [ ]:
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
fold_splits = list(skf.split(np.arange(len(y_raw)), y_raw))

print(f'StratifiedKFold - {K_FOLDS} folds over {len(y_raw)} images')
for fi, (tr, va) in enumerate(fold_splits):
    cnt = np.bincount(y_raw[va], minlength=NUM_CLASSES)
    print(f'  Fold {fi+1}: train={len(tr):5d}  val={len(va):4d}  val class dist={cnt.tolist()}')


## 7 · CNN Architecture — `RegressionCNN` (from `260305_cnn_regression_analysis.ipynb`)

Copied verbatim from the reference notebook: **6 conv blocks** (16→32→64→128→256→512 channels — the first
5 blocks each have two Conv-BN-ReLU pairs, the last block has a single Conv-BN-ReLU), each followed by a
`MaxPool2d(2,2)`, **flattened** (not global-average-pooled) into a 3-layer FC head
(`flat → flat/2 → flat/4 → 1`), with an **ELU** activation on the final output (so predictions are only
soft-floored near −1, not hard-clipped — there's no sigmoid-to-[0,10] rescaling here, matching the
reference notebook's commented-out choice not to use one).

In [ ]:
class RegressionCNN(nn.Module):
    '''Custom 2D CNN for regression — identical architecture to
    260305_cnn_regression_analysis.ipynb's RegressionCNN.'''
    def __init__(self, input_channels=3, img_height=64, img_width=64, depth=16):
        super(RegressionCNN, self).__init__()

        self.conv_layers = nn.Sequential(
            # Block 1: in -> depth
            nn.Conv2d(input_channels, depth, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth), nn.ReLU(),
            nn.Conv2d(depth, depth, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth), nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Block 2: depth -> depth*2
            nn.Conv2d(depth, depth * 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 2), nn.ReLU(),
            nn.Conv2d(depth * 2, depth * 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 2), nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Block 3: depth*2 -> depth*4
            nn.Conv2d(depth * 2, depth * 4, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 4), nn.ReLU(),
            nn.Conv2d(depth * 4, depth * 4, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 4), nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Block 4: depth*4 -> depth*8
            nn.Conv2d(depth * 4, depth * 8, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 8), nn.ReLU(),
            nn.Conv2d(depth * 8, depth * 8, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 8), nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Block 5: depth*8 -> depth*16
            nn.Conv2d(depth * 8, depth * 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 16), nn.ReLU(),
            nn.Conv2d(depth * 16, depth * 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 16), nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Block 6: depth*16 -> depth*32 (single conv, matching the reference notebook)
            nn.Conv2d(depth * 16, depth * 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(depth * 32), nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )

        # 6 maxpools -> spatial dims shrink by 2^6=64x; flattened_size generalizes to any
        # img_height/img_width divisible by 64 (the reference notebook used 64x64 -> 1x1 spatial)
        self.flattened_size = depth * img_height * img_width // 128
        print(f'  Flattened size after conv layers: {self.flattened_size}')

        self.fc_layers = nn.Sequential(
            nn.Linear(self.flattened_size, self.flattened_size // 2), nn.ReLU(),
            nn.Linear(self.flattened_size // 2, self.flattened_size // 4), nn.ReLU(),
            nn.Linear(self.flattened_size // 4, 1),   # single output for regression
            nn.ELU(),
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x


m = RegressionCNN(input_channels=3, img_height=IMG_SIZE[0], img_width=IMG_SIZE[1], depth=CNN_DEPTH)
n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f'RegressionCNN (in_ch=3, {IMG_SIZE[0]}x{IMG_SIZE[1]}, depth={CNN_DEPTH}): {n_params:,} trainable params')
del m


## 8 · Dataset & Augmentation

Same augmentation recipe as the reference notebook's `ImageRegressionDataset`/`train_transform`: random
horizontal + vertical flips, a random 90°-multiple rotation, `Normalize(mean=0.5, std=0.5)` on every
channel, and Gaussian noise added to the *training* targets only (`TARGET_NOISE_STD`) as a regularizer.
Resizing already happened during DNG demosaicing (Section 3), so it isn't repeated here.

In [ ]:
class LABRegressionDataset(Dataset):
    '''(H,W,3) uint8 LAB stack -> augmented, normalized CHW tensor, regression target = score 1..9.
    Mirrors ImageRegressionDataset from the reference notebook, adapted to work directly on the
    precomputed LAB stacks instead of re-loading/re-resizing images from disk each epoch.'''
    def __init__(self, stacks_uint8, labels, augment=False, target_noise_std=0.0):
        self.imgs = torch.from_numpy(stacks_uint8.astype(np.float32) / 255.0).permute(0, 3, 1, 2)
        self.scores = torch.from_numpy(IDX_TO_SCORE[labels]).unsqueeze(1)   # (N,1)
        self.labels = labels
        self.augment = augment
        self.target_noise_std = target_noise_std
        self.normalize = transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img = self.imgs[idx].clone()
        score = self.scores[idx].clone()

        if self.augment:
            if random.random() < 0.5: img = img.flip(2)   # H flip
            if random.random() < 0.5: img = img.flip(1)   # V flip
            k = random.choice([0, 1, 2, 3])
            if k: img = torch.rot90(img, k=k, dims=[1, 2])

        img = self.normalize(img)

        if self.target_noise_std > 0:
            score = score + torch.randn_like(score) * self.target_noise_std

        return img, score, self.labels[idx]


## 9 · Training Engine

Same loss/optimizer/scheduler/early-stopping recipe as the reference notebook: `MSELoss`, Adam
(`lr=1e-3`, `weight_decay=0`), `ReduceLROnPlateau(factor=0.1, patience=5)` stepped on **validation loss**,
and early stopping (patience 12) tracking the **best validation loss** — not R² — exactly as in
`260305_cnn_regression_analysis.ipynb`'s training loop. Every fold's OOF predictions are cached to disk
(`dng_cnn_cache/oof_{tag}.pkl`) so a completed encoding is never retrained unless `FORCE_RETRAIN = True`.

In [ ]:
def compute_metrics(preds, targets):
    preds   = np.array(preds, dtype=np.float32).flatten()
    targets = np.array(targets, dtype=np.float32).flatten()
    return {
        'R2':       r2_score(targets, preds),
        'RMSE':     float(np.sqrt(mean_squared_error(targets, preds))),
        'MAE':      mean_absolute_error(targets, preds),
        'Spearman': spearmanr(targets, preds).correlation,
        'Kendall':  kendalltau(targets, preds).correlation,
    }


def get_preds_and_targets(model, loader, dev):
    model.eval()
    preds_list, scores_list = [], []
    with torch.no_grad():
        for img, score, _ in loader:
            img = img.to(dev)
            pred = model(img)
            preds_list.append(pred.cpu().float())
            scores_list.append(score.cpu().float())
    preds  = torch.cat(preds_list).numpy().flatten()
    scores = torch.cat(scores_list).numpy().flatten()
    return preds, scores


def train_one_fold(in_ch, fold_idx, tr_idx, va_idx, stacks, tag):
    ds_tr = LABRegressionDataset(stacks[tr_idx], y_raw[tr_idx], augment=True, target_noise_std=TARGET_NOISE_STD)
    ds_va = LABRegressionDataset(stacks[va_idx], y_raw[va_idx], augment=False, target_noise_std=0.0)
    tr_loader = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    va_loader = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    torch.manual_seed(SEED + fold_idx)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED + fold_idx)

    model = RegressionCNN(input_channels=in_ch, img_height=IMG_SIZE[0], img_width=IMG_SIZE[1],
                           depth=CNN_DEPTH).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=LR_INIT, weight_decay=0.0)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=LR_FACTOR,
                                                      patience=PATIENCE_LR)

    best_val_loss, wait_es = float('inf'), 0
    best_path = os.path.join(CKPT_DIR, f'best_{tag}_fold{fold_idx+1}.pt')
    hist = {'train_loss': [], 'val_loss': [], 'val_r2': []}

    t0 = time.time()
    for epoch in range(EPOCHS):
        model.train()
        ep_loss = []
        for img, score, _ in tr_loader:
            img, score = img.to(device), score.to(device)
            optimizer.zero_grad()
            out = model(img)
            loss = criterion(out, score)
            loss.backward()
            optimizer.step()
            ep_loss.append(loss.item())
        tr_loss = float(np.mean(ep_loss))

        val_preds, val_scores = get_preds_and_targets(model, va_loader, device)
        val_loss = float(np.mean((val_preds - val_scores) ** 2))
        m = compute_metrics(val_preds, val_scores)

        lr_before = optimizer.param_groups[0]['lr']
        scheduler.step(val_loss)
        lr_after = optimizer.param_groups[0]['lr']

        hist['train_loss'].append(tr_loss); hist['val_loss'].append(val_loss); hist['val_r2'].append(m['R2'])

        marker = ''
        if val_loss < best_val_loss:
            best_val_loss, wait_es = val_loss, 0
            torch.save(model.state_dict(), best_path)
            marker = f'  best val_loss={best_val_loss:.4f}'
        else:
            wait_es += 1
        if lr_after < lr_before:
            marker += f'  LR->{lr_after:.1e}'

        if (epoch + 1) % 10 == 0 or marker:
            print(f'    Ep {epoch+1:4d} | train_loss={tr_loss:.4f} val_loss={val_loss:.4f} '
                  f'R2={m["R2"]:.3f} rho={m["Spearman"]:.3f}{marker}')

        if wait_es >= PATIENCE_ES:
            print(f'    Early stop @ epoch {epoch+1} (best val_loss={best_val_loss:.4f})')
            break

    elapsed = time.time() - t0
    model.load_state_dict(torch.load(best_path, map_location=device))
    oof_p, oof_t = get_preds_and_targets(model, va_loader, device)
    del model
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return oof_p, oof_t, hist, elapsed


def train_cnn_cv(stacks, in_ch, tag):
    '''Runs the full 5-fold CV for one encoding configuration, with disk caching.'''
    cache_path = os.path.join(CACHE_DIR, f'oof_{tag}.pkl')
    if os.path.exists(cache_path) and not FORCE_RETRAIN:
        with open(cache_path, 'rb') as f:
            cached = pickle.load(f)
        print(f"[{tag}] loaded cached OOF results  "
              f"(R2={cached['oof_metrics']['R2']:.4f}, fit_time={cached['fit_time']:.1f}s)")
        return cached

    n = len(y_raw)
    oof_pred = np.zeros(n, dtype=np.float32)
    fold_metrics, fold_histories = [], []
    t0 = time.time()
    print(f'[{tag}] training {K_FOLDS} folds (in_ch={in_ch}) ...')
    for fi, (tr_idx, va_idx) in enumerate(fold_splits):
        print(f'  Fold {fi+1}/{K_FOLDS}')
        oof_p, oof_t, hist, elapsed = train_one_fold(in_ch, fi, tr_idx, va_idx, stacks, tag)
        oof_pred[va_idx] = oof_p
        fold_metrics.append(compute_metrics(oof_p, oof_t))
        fold_histories.append(hist)
        print(f'    Fold {fi+1} done in {elapsed/60:.1f} min')
    fit_time = time.time() - t0

    oof_metrics = compute_metrics(oof_pred, y)
    result = {'tag': tag, 'oof_pred': oof_pred, 'oof_metrics': oof_metrics,
              'fold_metrics': pd.DataFrame(fold_metrics), 'fold_histories': fold_histories,
              'fit_time': fit_time, 'in_ch': in_ch}
    with open(cache_path, 'wb') as f:
        pickle.dump(result, f)

    print(f"[{tag}] fit_time={fit_time/60:.1f}min  OOF R2={oof_metrics['R2']:.4f}  "
          f"RMSE={oof_metrics['RMSE']:.4f}  MAE={oof_metrics['MAE']:.4f}  "
          f"Spearman={oof_metrics['Spearman']:.4f}  Kendall={oof_metrics['Kendall']:.4f}")
    return result


## 10 · Encoding-Method Ablation — sRGB vs Linear vs ACES

`LAB` color space and `Statistical (K=2.0σ)` filter are held fixed; only the RAW→RGB encoding varies.
For each encoding: demosaic all DNGs (cached), apply the statistical filter, convert to LAB, then run the
full 5-fold CV training with `RegressionCNN`.

In [ ]:
results = {}   # keyed by encoding name

for encoding in ENCODING_METHODS:
    print(f'\n=== Encoding: {encoding} ===')
    rgb_imgs   = load_encoded_dataset(flat_paths, IMG_SIZE, encoding)
    filt_imgs  = apply_stat_filter(rgb_imgs, tag=encoding)
    lab_stack  = build_lab_stack(filt_imgs, tag=encoding)
    tag = f'{encoding}_LAB_statK2'
    results[encoding] = train_cnn_cv(lab_stack, in_ch=3, tag=tag)


In [ ]:
enc_rows = []
for encoding in ENCODING_METHODS:
    r = results[encoding]
    enc_rows.append({'Encoding': encoding, 'Fit time (min)': round(r['fit_time'] / 60, 1),
                      **r['oof_metrics']})

enc_table = pd.DataFrame(enc_rows).set_index('Encoding')
enc_table = enc_table[['R2', 'RMSE', 'MAE', 'Spearman', 'Kendall', 'Fit time (min)']]
print(f'Encoding-Method Ablation (LAB color space, {FIXED_FILTER_NAME} filter, RegressionCNN, OOF metrics)')
enc_table.round(4)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = plt.cm.plasma(np.linspace(0.15, 0.85, len(ENCODING_METHODS)))

r2_vals = [results[e]['oof_metrics']['R2'] for e in ENCODING_METHODS]
bars = axes[0].bar(ENCODING_METHODS, r2_vals, color=colors, edgecolor='black')
axes[0].set_title('RegressionCNN R2 by RAW->RGB Encoding'); axes[0].set_ylabel('OOF R2')
for b, v in zip(bars, r2_vals):
    axes[0].text(b.get_x() + b.get_width()/2, v, f'{v:.3f}', ha='center',
                 va='bottom' if v >= 0 else 'top', fontsize=9)
axes[0].grid(axis='y', alpha=0.3)

rho_vals = [results[e]['oof_metrics']['Spearman'] for e in ENCODING_METHODS]
bars = axes[1].bar(ENCODING_METHODS, rho_vals, color=colors, edgecolor='black')
axes[1].set_title('RegressionCNN Spearman \u03c1 by RAW->RGB Encoding'); axes[1].set_ylabel('OOF Spearman \u03c1')
for b, v in zip(bars, rho_vals):
    axes[1].text(b.get_x() + b.get_width()/2, v, f'{v:.3f}', ha='center',
                 va='bottom' if v >= 0 else 'top', fontsize=9)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Encoding-Method Ablation (LAB + Statistical K=2.0\u03c3, RegressionCNN)', y=1.03)
plt.tight_layout(); savefig('01_encoding_ablation_cnn.png'); plt.show()

BEST_ENC = enc_table['R2'].idxmax()
print(f'Best encoding: {BEST_ENC}  (R2={enc_table.loc[BEST_ENC, "R2"]:.4f})')


## 11 · Predicted vs Actual, Residuals & Training Curves — Best Encoding

In [ ]:
best_result = results[BEST_ENC]

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
pred, actual = best_result['oof_pred'], y

axes[0].scatter(actual, pred, alpha=0.5, s=20, color='steelblue', edgecolor='k', linewidth=0.2)
lims = [min(actual.min(), pred.min()) - 0.3, max(actual.max(), pred.max()) + 0.3]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Ideal (y=x)')
axes[0].set_xlim(lims); axes[0].set_ylim(lims)
axes[0].set_xlabel('Actual Score'); axes[0].set_ylabel('Predicted Score')
axes[0].set_title(f"{BEST_ENC} — Predicted vs Actual (R2={best_result['oof_metrics']['R2']:.3f})")
axes[0].legend(); axes[0].grid(alpha=0.3)

residuals = pred - actual
axes[1].scatter(actual, residuals, alpha=0.5, s=20, color='indianred', edgecolor='k', linewidth=0.2)
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_xlabel('Actual Score'); axes[1].set_ylabel('Residual (pred - actual)')
axes[1].set_title(f'{BEST_ENC} — Residuals (mean={residuals.mean():.3f}, std={residuals.std():.3f})')
axes[1].grid(alpha=0.3)

plt.tight_layout(); savefig('02_best_encoding_pred_vs_actual_residuals.png'); plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fold_colors = plt.cm.tab10(np.arange(K_FOLDS))
for fi, hist in enumerate(best_result['fold_histories']):
    axes[0].plot(hist['val_loss'], color=fold_colors[fi], label=f'Fold {fi+1}', alpha=0.8)
    axes[1].plot(hist['val_r2'],   color=fold_colors[fi], label=f'Fold {fi+1}', alpha=0.8)
axes[0].set_title(f'{BEST_ENC} — Validation Loss per Fold'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE (val)')
axes[1].set_title(f'{BEST_ENC} — Validation R2 per Fold');   axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('R2')
for ax in axes: ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); savefig('03_best_encoding_training_curves.png'); plt.show()


## 12 · Sample Images Across Encodings (Visual Sanity Check)

Before trusting the metrics above, look at the same physical zucchini rendered through each encoding
(post statistical-filter) — this is the fastest way to catch a broken color matrix, a bad exposure
normalization, or a filter that isn't isolating the produce the way it did on the pre-cropped TIFFs.

In [ ]:
sample_idx = 0
fig, axes = plt.subplots(1, len(ENCODING_METHODS) + 1, figsize=(4 * (len(ENCODING_METHODS) + 1), 4))

raw_preview = load_encoded_dataset(flat_paths, IMG_SIZE, 'sRGB')[sample_idx]
axes[0].imshow(raw_preview); axes[0].set_title('sRGB (unfiltered)'); axes[0].axis('off')

for ax, encoding in zip(axes[1:], ENCODING_METHODS):
    rgb_imgs = load_encoded_dataset(flat_paths, IMG_SIZE, encoding)
    filt = stat_filter(rgb_imgs[sample_idx], k=K_STAT)
    ax.imshow(filt['out'])
    ax.set_title(f"{encoding} [filtered]\nkept={filt['kept_pct']:.0f}%")
    ax.axis('off')

plt.suptitle('Encoding Comparison — Example Image (Statistical K=2.0\u03c3 filter applied)', y=1.05)
plt.tight_layout(); savefig('04_encoding_comparison_example.png'); plt.show()


## 13 · Summary & Key Takeaways

In [ ]:
print('=' * 88)
print('SUMMARY — DNG RAW Encoding Ablation (LAB + Statistical K=2.0\u03c3, RegressionCNN)')
print('=' * 88)
print(f'\nBest encoding : {BEST_ENC}')
bm = results[BEST_ENC]['oof_metrics']
print(f"  R2={bm['R2']:.4f}  RMSE={bm['RMSE']:.4f}  MAE={bm['MAE']:.4f}  "
      f"Spearman={bm['Spearman']:.4f}  Kendall={bm['Kendall']:.4f}")

print('\nAll encodings:')
for encoding in ENCODING_METHODS:
    m = results[encoding]['oof_metrics']
    print(f"  {encoding:8s} R2={m['R2']:.4f}  RMSE={m['RMSE']:.4f}  "
          f"Spearman={m['Spearman']:.4f}  fit_time={results[encoding]['fit_time']/60:.1f}min")

print('\nAll figures saved to:', os.path.abspath(OUT_DIR))
print('All cached decodes/OOF results/checkpoints saved to:',
      os.path.abspath(CACHE_DIR), 'and', os.path.abspath(CKPT_DIR))
print('=' * 88)


### Notes for interpretation

- **Model swap vs. the earlier DNG-ablation draft**: this notebook now trains the exact `RegressionCNN`
  from `260305_cnn_regression_analysis.ipynb` (6 conv blocks, flatten+FC, ELU output) with its own loss/
  optimizer/scheduler/early-stopping recipe and target-noise regularization, rather than the GAP-based
  `CNNRegressor` used previously. If you want to compare the two models on the *same* DNG data, just
  swap `RegressionCNN` back for the earlier architecture in Section 7 — everything else in this notebook
  (data loading, filter, color space, CV splits, ablation loop) is architecture-agnostic.
- **If the three encodings land close together** (small R²/Spearman gap), it suggests the model is largely
  learning to normalize away the tone-curve/gamut differences on its own — plausible given BatchNorm
  absorbs a lot of global intensity-scale differences.
- **If ACES or Linear clearly wins**, check whether that's because the encoding preserves more
  discriminative color information in the mid-tones/highlights that the color-corrected sRGB TIFF
  pipeline was compressing away — versus an artifact of the percentile-based exposure normalization
  landing differently per image (check the residuals plot in Section 11 for score-dependent bias, which
  would point to the latter).
- **If sRGB clearly wins**, that's a reasonable prior: it's the encoding closest to what the existing
  TIFF pipeline already produces.
- **Re-run the ACES path with a real ACES/OCIO transform** before treating a strong ACES result as
  final — see the caveat in Section 3.
- Compare this notebook's master table against `zucchini_cnn_colorspace_filtering_ablation.ipynb`'s
  best result to see how much of any performance gap is attributable to skipping the nocc crop/mask step
  versus the encoding itself, versus the model architecture change.
